# Road Safety Severity Prediction — Machine Learning Pipeline
This notebook contains the complete data preprocessing, cross-validation, hyperparameter tuning, model training, and evaluation code for predicting whether a road collision results in a **Severe** (Fatal or Serious) outcome vs. a **Slight** outcome.

We use provisional 2025 STATS19 police-reported data from the UK Department for Transport (DfT).

## 1. Import Libraries

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    auc,
    f1_score,
    roc_auc_score
)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid")

## 2. Load Datasets
We load all three related tables (collisions, casualties, and vehicles) to build a multi-dimensional feature space.

In [ ]:
print("Loading datasets...")
collision_path = "dft-road-casualty-statistics-collision-provisional-2025 (1).csv"
casualty_path = "dft-road-casualty-statistics-casualty-provisional-2025.csv"
vehicle_path = "dft-road-casualty-statistics-vehicle-provisional-2025.csv"

df = pd.read_csv(collision_path, low_memory=False)
cas_df = pd.read_csv(casualty_path, low_memory=False)
veh_df = pd.read_csv(vehicle_path, low_memory=False)

print(f"Collisions: {df.shape}")
print(f"Casualties: {cas_df.shape}")
print(f"Vehicles:   {veh_df.shape}")

## 3. Data Cleaning & Mappings
We decode categorical attributes from raw integer codes to string labels for transparent encodings.

In [ ]:
# 1. Parse date/time
df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], format='%d/%m/%Y %H:%M', errors='coerce')
df = df.dropna(subset=['datetime'])
df['hour'] = df['datetime'].dt.hour

# 2. Mappings dictionaries
severity_map = {1: "Fatal", 2: "Serious", 3: "Slight"}
day_map = {1: "Sunday", 2: "Monday", 3: "Tuesday", 4: "Wednesday", 5: "Thursday", 6: "Friday", 7: "Saturday"}
urban_rural_map = {1: "Urban", 2: "Rural", 3: "Unallocated"}
light_map = {
    1: "Daylight",
    4: "Darkness - lights lit",
    5: "Darkness - lights unlit",
    6: "Darkness - no lights",
    7: "Darkness - lights detail unknown"
}
weather_map = {
    1: "Fine no high winds",
    2: "Raining no high winds",
    3: "Snowing no high winds",
    4: "Fine + high winds",
    5: "Raining + high winds",
    6: "Snowing + high winds",
    7: "Fog or mist",
    8: "Other",
    9: "Unknown"
}
surface_map = {
    1: "Dry",
    2: "Wet or damp",
    3: "Snow",
    4: "Frost or ice",
    5: "Flood over 3cm deep",
    6: "Oil or wet mud",
    7: "Road sign"
}

# Clean missing codes (-1) to NaN
df['urban_or_rural_area'] = df['urban_or_rural_area'].replace(-1, np.nan)
df['light_conditions'] = df['light_conditions'].replace(-1, np.nan)
df['weather_conditions'] = df['weather_conditions'].replace(-1, np.nan)
df['road_surface_conditions'] = df['road_surface_conditions'].replace(-1, np.nan)
df['speed_limit'] = df['speed_limit'].replace(-1, np.nan)

# Decode
df['collision_severity_label'] = df['collision_severity'].map(severity_map)
df['day_of_week_label'] = df['day_of_week'].map(day_map)
df['urban_or_rural_label'] = df['urban_or_rural_area'].map(urban_rural_map)
df['light_conditions_label'] = df['light_conditions'].map(light_map)
df['weather_conditions_label'] = df['weather_conditions'].map(weather_map)
df['road_surface_conditions_label'] = df['road_surface_conditions'].map(surface_map)

# Drop rows where target severity is null
df = df.dropna(subset=['collision_severity_label'])

# Create target variable: Severe (1) vs Slight (0)
df['is_severe'] = df['collision_severity'].isin([1, 2]).astype(int)

## 4. Advanced Feature Engineering
We aggregate demographics and road user types from vehicle and casualty details up to the collision level to improve predictive performance:
- **Vulnerable Road Users (VRUs)**: Flag the involvement of motorcycles, pedal cycles (bicycles), and pedestrians.
- **Vehicle Classes**: Flag HGV or Bus involvement.
- **Age Profiles**: Extract minimum and maximum age bounds for both drivers and casualties in the collision.

In [ ]:
print("Engineering vehicle features...")
# Motorcycles: codes 2, 3, 4, 5, 97
# Pedal cycles: code 1
# HGV or Bus: codes 11, 20, 21
veh_df['is_motorcycle'] = veh_df['vehicle_type'].isin([2, 3, 4, 5, 97]).astype(int)
veh_df['is_pedal_cycle'] = (veh_df['vehicle_type'] == 1).astype(int)
veh_df['is_hgv_or_bus'] = veh_df['vehicle_type'].isin([11, 20, 21]).astype(int)
veh_df['age_of_driver'] = veh_df['age_of_driver'].replace(-1, np.nan)

veh_agg = veh_df.groupby('collision_index').agg({
    'is_motorcycle': 'max',
    'is_pedal_cycle': 'max',
    'is_hgv_or_bus': 'max',
    'age_of_driver': ['min', 'max']
})
veh_agg.columns = ['has_motorcycle', 'has_pedal_cycle', 'has_hgv_or_bus', 'driver_age_min', 'driver_age_max']
veh_agg = veh_agg.reset_index()

print("Engineering casualty features...")
# Pedestrians: casualty class = 3
cas_df['is_pedestrian'] = (cas_df['casualty_class'] == 3).astype(int)
cas_df['age_of_casualty'] = cas_df['age_of_casualty'].replace(-1, np.nan)

cas_agg = cas_df.groupby('collision_index').agg({
    'is_pedestrian': 'max',
    'age_of_casualty': ['min', 'max']
})
cas_agg.columns = ['has_pedestrian', 'casualty_age_min', 'casualty_age_max']
cas_agg = cas_agg.reset_index()

print("Merging features into collisions...")
model_df = df.merge(veh_agg, on='collision_index', how='left')
model_df = model_df.merge(cas_agg, on='collision_index', how='left')

# Fill missing flags with 0
for col in ['has_motorcycle', 'has_pedal_cycle', 'has_hgv_or_bus', 'has_pedestrian']:
    model_df[col] = model_df[col].fillna(0).astype(int)

print(f"Completed feature engineering. Final shape: {model_df.shape}")

## 5. Splits & Preprocessing
We partition the dataset into an 80% train and 20% holdout test set (stratified by target variable `is_severe`), and design a Scikit-Learn `ColumnTransformer` preprocessing pipeline.

In [ ]:
NUM_FEATURES = [
    'speed_limit', 'hour', 'number_of_vehicles',
    'driver_age_min', 'driver_age_max',
    'casualty_age_min', 'casualty_age_max'
]
CAT_FEATURES = [
    'urban_or_rural_label',
    'light_conditions_label',
    'weather_conditions_label',
    'road_surface_conditions_label',
    'day_of_week_label'
]
BIN_FEATURES = ['has_motorcycle', 'has_pedal_cycle', 'has_hgv_or_bus', 'has_pedestrian']

X = model_df[NUM_FEATURES + CAT_FEATURES + BIN_FEATURES]
y = model_df['is_severe']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")

In [ ]:
# Preprocessors
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, NUM_FEATURES),
        ('cat', categorical_transformer, CAT_FEATURES)
    ],
    remainder='passthrough' # Leave binary features as-is
)

## 6. Model Training & Validation
We evaluate Random Forest and XGBoost with **5-Fold Stratified Cross-Validation**, using target class weighting to counter class imbalance.

In [ ]:
# Compute scale_pos_weight ratio
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count
print(f"scale_pos_weight: {scale_pos_weight:.3f}")

models = {
    'RandomForest': RandomForestClassifier(
        n_estimators=150, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200, max_depth=6, scale_pos_weight=scale_pos_weight,
        learning_rate=0.05, random_state=42, n_jobs=-1, eval_metric='logloss'
    )
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}
best_pipeline = None
best_f1 = 0
best_name = ""

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    # Cross-validation
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='f1', n_jobs=-1)
    mean_f1 = np.mean(scores)
    print(f"{name} Mean CV F1-Score: {mean_f1:.4f}")
    
    pipeline.fit(X_train, y_train)
    
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)
    print(f"{name} Test F1-Score: {f1:.4f} | Test ROC-AUC: {roc_auc:.4f}")
    
    results[name] = {
        'pipeline': pipeline,
        'test_f1': f1,
        'test_roc_auc': roc_auc,
        'y_pred': y_pred,
        'y_prob': y_prob
    }
    
    if mean_f1 > best_f1:
        best_f1 = mean_f1
        best_pipeline = pipeline
        best_name = name

print(f"\nSelected Best Model: {best_name}")

## 7. Model Evaluation
We output classification metrics, confusion matrices, ROC curves, and Gini feature importances for our best-performing model.

In [ ]:
best_y_pred = results[best_name]['y_pred']
best_y_prob = results[best_name]['y_prob']

print("Classification Report:")
print(classification_report(y_test, best_y_pred, target_names=['Slight', 'Severe']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, best_y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Slight', 'Severe'], 
            yticklabels=['Slight', 'Severe'])
plt.title(f'{best_name} Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, best_y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title(f'{best_name} ROC Curve')
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Feature Importance
fitted_preprocessor = best_pipeline.named_steps['preprocessor']
fitted_classifier = best_pipeline.named_steps['classifier']

cat_encoder = fitted_preprocessor.named_transformers_['cat'].named_steps['onehot']
encoded_cat_features = cat_encoder.get_feature_names_out(CAT_FEATURES).tolist()
all_features = NUM_FEATURES + encoded_cat_features + BIN_FEATURES

importances = pd.Series(fitted_classifier.feature_importances_, index=all_features).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=importances.head(12).values, y=importances.head(12).index, palette="viridis")
plt.title('Top 12 Features by Gini Importance')
plt.xlabel('Relative Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## 8. Export Model Pipeline
We save the final preprocessor and model state to `road_safety_model.joblib`.

In [ ]:
model_file = 'road_safety_model.joblib'
joblib.dump(best_pipeline, model_file)
print(f"Successfully saved best model pipeline to {model_file}!")